# **01 - Proceso de Limpieza de Datos**  
*Por Andrés Calderón Bernal*

## Descripción del caso

Más de cinco mil aspirantes a aprendiz del Servicio Nacional de Aprendizaje (SENA), en Colombia, presentaron una prueba escrita en las áreas de Razonamiento Cuantitativo, Comprensión Lectora e Inglés. 

Además de los resultados obtenidos, se registró información demográfica básica: sexo, ciudad de residencia y fecha de nacimiento. Estos datos se encuentran almacenados en un archivo CSV y requieren un proceso previo de validación y limpieza antes de ser utilizados para análisis exploratorio.

No se establece si la ciudad de residencia es la misma donde está la sede SENA en la que el aspirante pretende cursar un programa.

La edad mínima para cursar programas en el SENA, en el caso de los programas virtuales, es de catorce años. No hay edad máxima para entrar. Los exámenes se realizaron en marzo de 2025.

En este notebook se evaluará la calidad de los datos, identificando valores nulos, inconsistencias y registros fuera de los rangos esperados, con el fin de construir un dataset confiable para análisis posterior.

## Descripción de las variables

El dataset contiene las siguientes variables:

- **Sexo**: variable categórica nominal (Femenino, Masculino, NR/Otro (para identidades no binarias)).
- **Ciudad**: variable categórica nominal correspondiente a la ciudad de residencia.
- **Fecha de Nacimiento**: variable temporal (datetime), restringida para registros posteriores a marzo de 2011.
- **Nota Razonamiento Cuantitativo**
- **Nota Comprensión Lectora**
- **Nota Inglés**

Las tres columnas de notas corresponden a variables cuantitativas continuas en una escala de 1.0 a 5.0, y deben tener solo un número decimal.


**Aclaración**: Un *dataset* es el conjunto completo de datos recopilados para un propósito específico, mientras que las variables son las características o atributos individuales que describen cada observación dentro de ese conjunto. Por su parte, un *dataframe* es una estructura tabular que organiza el dataset en filas y columnas; en este contexto, las columnas representan las variables y las filas corresponden a cada registro.

## **1.** Exploración inicial de estructura y valores nulos

In [1]:
import pandas as pd
import numpy as np

en_bruto = pd.read_csv('../datos/datos_en_bruto.csv')
print(f'Dimensiones del dataset: {en_bruto.shape}') #Arroja las dimensiones del dataframe
print(f'Columnas: {list(en_bruto.columns)}') #Arroja las columnas del dataframe
en_bruto.info()
en_bruto.head()

Dimensiones del dataset: (5023, 6)
Columnas: ['Sexo', 'Ciudad', 'Fecha de Nacimiento', 'Nota Razonamiento Cuantitativo', 'Nota Comprension Lectora', 'Nota Ingles']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5023 entries, 0 to 5022
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Sexo                            5021 non-null   object 
 1   Ciudad                          5017 non-null   object 
 2   Fecha de Nacimiento             5011 non-null   object 
 3   Nota Razonamiento Cuantitativo  5022 non-null   float64
 4   Nota Comprension Lectora        5022 non-null   float64
 5   Nota Ingles                     5021 non-null   float64
dtypes: float64(3), object(3)
memory usage: 235.6+ KB


,Sexo,Ciudad,Fecha de Nacimiento,Nota Razonamiento Cuantitativo,Nota Comprension Lectora,Nota Ingles
0,Femenino,Bucaramanga,1997-11-27,2.82,1.39,3.72
1,NaN,Maicao,2002-04-06,2.12,2.80,2.52
2,Masculino,Maicao,1998-03-11,2.41,1.99,2.05
3,M,Sogamoso,1997-11-03,3.23,2.15,2.30
4,Femenino,Maicao,2001-09-07,2.49,1.50,3.23


El dataset en el archivo csv *datos_en_bruto*, en la carpeta *datos*, contiene 5023 registros y seis variables. A partir de la inspección inicial, se observa lo siguiente:

- Tres variables categóricas almacenadas como tipo `object`:  
  **Sexo**, **Ciudad** y **Fecha de Nacimiento**.
- Tres variables numéricas de tipo `float64` correspondientes a las notas.
- Existen valores faltantes en todas las columnas, si bien en proporciones ínfimas.

Se destaca que la variable **Fecha de Nacimiento** se encuentra almacenada como texto (`object`), por lo que, a su tiempo, será necesario convertirla a formato datetime para su validación y posterior transformación en edad.

Por otro lado, las notas presentan dos números decimales en vez de uno. Dado que solo deben tener una cifra decimal, se redondea los valores en estas columnas.

In [2]:
notas = ["Nota Razonamiento Cuantitativo", "Nota Comprension Lectora", "Nota Ingles"]
en_bruto[notas] = en_bruto[notas].round(1) #Redondear a 1 decimal

## **2.** Tratamiento de valores nulos

In [3]:
nulos_por_columna = en_bruto.isnull().sum() #Arroja la cantidad de nulos por columna
print("Valores nulos por columna:")
print(nulos_por_columna)

porcentaje_nulos = (en_bruto.isnull().mean() * 100).round(2) #Arroja el porcentaje de nulos por columna
print("Porcentaje de valores nulos por columna:")
print(porcentaje_nulos)

filas_con_nulos = en_bruto[en_bruto.isnull().any(axis=1)] #Registros del dataset que contienen al menos un nulo
print(f'Número de filas con valores nulos: {len(filas_con_nulos)}')
filas_con_nulos.head()


Valores nulos por columna:
Sexo                               2
Ciudad                             6
Fecha de Nacimiento               12
Nota Razonamiento Cuantitativo     1
Nota Comprension Lectora           1
Nota Ingles                        2
dtype: int64
Porcentaje de valores nulos por columna:
Sexo                              0.04
Ciudad                            0.12
Fecha de Nacimiento               0.24
Nota Razonamiento Cuantitativo    0.02
Nota Comprension Lectora          0.02
Nota Ingles                       0.04
dtype: float64
Número de filas con valores nulos: 22


,Sexo,Ciudad,Fecha de Nacimiento,Nota Razonamiento Cuantitativo,Nota Comprension Lectora,Nota Ingles
1,NaN,Maicao,2002-04-06,2.1,2.8,2.5
183,Masculino,NaN,2004-02-02,2.6,1.2,2.6
196,NaN,Barrancabermeja,2004-10-02,2.0,1.5,2.0
775,Masculino,Tauramena,NaN,3.2,1.3,15.0
786,Femenino,Bucaramanga,NaN,4.5,1.9,1.9


Se identificaron **22** registros con al menos un valor nulo; cifra ínfima, tomando en cuenta que el dataframe está compuesto por más de cinco mil registros.

La proporción de valores nulos por columna es inferior al 0.25% en todos los casos, siendo la columna **Fecha de Nacimiento** la que presenta mayor frecuencia de datos faltantes (12, un 0.24% del total).

Dado que la cantidad de registros incompletos es marginal y no compromete la representatividad del conjunto de datos, se opta por eliminar los registros que contienen valores nulos, y así evitar introducir sesgos innecesarios.

In [4]:
en_bruto = en_bruto.dropna() #Eliminación de registros con valores nulos

#Verificar que sí se borraron tales registros
print(f'Dimensiones después de eliminar nulos: {en_bruto.shape}')
print(f'Valores nulos restantes: {en_bruto.isnull().sum().sum()}')
en_bruto.info()

Dimensiones después de eliminar nulos: (5001, 6)
Valores nulos restantes: 0
<class 'pandas.core.frame.DataFrame'>
Index: 5001 entries, 0 to 5022
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Sexo                            5001 non-null   object 
 1   Ciudad                          5001 non-null   object 
 2   Fecha de Nacimiento             5001 non-null   object 
 3   Nota Razonamiento Cuantitativo  5001 non-null   float64
 4   Nota Comprension Lectora        5001 non-null   float64
 5   Nota Ingles                     5001 non-null   float64
dtypes: float64(3), object(3)
memory usage: 273.5+ KB


Tras la depuración, el número de registros se redujo a **5001**.

## **3.** Validación de la columna Fecha de Nacimiento

La variable Fecha de Nacimiento pasa de ser tipo `object` a formato `datetime` para permitir su validación.

Se verifica la existencia de fechas mal formateadas, con el propósito de identificar registros con fechas inválidas como valores nulos NaT (Not a time).

In [5]:
en_bruto["Fecha de Nacimiento"] = pd.to_datetime( #Convierte la variable a formato datetime
    en_bruto["Fecha de Nacimiento"],
    errors="coerce" #Marca fechas inválidas como NaT
)

print(f'Tipo: {en_bruto["Fecha de Nacimiento"].dtype}') #Confirma que, en efecto, la columna sea datetime
print(f'Valores nulos resultantes: {en_bruto["Fecha de Nacimiento"].isnull().sum()}')

Tipo: datetime64[ns]
Valores nulos resultantes: 0



Tras la verificación, no se encontraron registros con fechas inválidas.

## **4.** Cálculo y validación de la variable Edad

La conversión a datetime de la variable Fecha de Nacimiento permite el cálculo de la variable Edad.

Dado que los exámenes fueron aplicados en marzo de 2025, la edad de los aspirantes debe calcularse con respecto a esa fecha y no a la fecha actual. Como no se proporcionó fecha exacta, se toma el primer día de dicho mes.

La edad mínima para cursar programas virtuales es de catorce años. Se busca si hay registros con edades menores a esa, o con valores atípicos.

In [6]:
fecha_examen = pd.Timestamp("2025-03-01")

# Cálculo de edad en años cumplidos
en_bruto["Edad"] = (
    (fecha_examen - en_bruto["Fecha de Nacimiento"]).dt.days // 365
)

# Verificación básica
print("Edad mínima:", en_bruto["Edad"].min())
print("Edad máxima:", en_bruto["Edad"].max())
print("Registros con edad menor a 14:",
      (en_bruto["Edad"] < 14).sum())

Edad mínima: -79
Edad máxima: 41
Registros con edad menor a 14: 2



Al realizar la verificación se obtuvieron los siguientes resultados:

- Edad mínima observada: -79 años

- Edad máxima observada: 41 años

- Registros con edad menor a 14 años: 2

La presencia de una edad negativa indica que existen registros con fechas de nacimiento posteriores a la fecha del examen, lo cual constituye una inconsistencia lógica en los datos.

Dado que los registros inconsistentes solo son dos, que representa una porción ínfima del total, se procede a eliminarlos del dataset, con el fin de garantizar coherencia y calidad analítica.

In [7]:
#Se suprimen los registros cuya edad es menor a 14 (incluidos valores negativos)
en_bruto = en_bruto[en_bruto["Edad"] >= 14]

print("Nueva edad mínima:", en_bruto["Edad"].min())
print("Registros restantes:", en_bruto.shape)

en_bruto.head()

Nueva edad mínima: 16
Registros restantes: (4999, 7)


,Sexo,Ciudad,Fecha de Nacimiento,Nota Razonamiento Cuantitativo,Nota Comprension Lectora,Nota Ingles,Edad
0,Femenino,Bucaramanga,1997-11-27,2.8,1.4,3.7,27
2,Masculino,Maicao,1998-03-11,2.4,2.0,2.0,26
3,M,Sogamoso,1997-11-03,3.2,2.2,2.3,27
4,Femenino,Maicao,2001-09-07,2.5,1.5,3.2,23
5,Femenino,Valledupar,1996-12-22,1.2,2.0,1.2,28


Tras esta depuración, el número de registros del dataset se redujo a **4999**. La edad mínima ahora es **16**.

## **5.** Validación de variables categóricas

In [8]:
#Evita errores por espacios invisibles o letras al inicio sin capitalizar
en_bruto["Sexo"] = (
    en_bruto["Sexo"]
    .str.strip()
    .str.title()  
)

en_bruto["Ciudad"] = (
    en_bruto["Ciudad"]
    .str.strip()
    .str.title()  
)

### **5.1** Variable Sexo

Se inspeccionan los valores únicos de la variable Sexo con el fin de identificar posibles inconsistencias tipográficas o categorías no previstas.

In [9]:
print(en_bruto["Sexo"].value_counts()) #Valores únicos presentes en la variable y su frecuencia absoluta

valores_aceptables_sexo = ["Masculino", "Femenino", "NR/Otro"] #Valores aceptables

print("Valores no aceptables restantes:",
      en_bruto[~en_bruto["Sexo"].isin(valores_aceptables_sexo)]["Sexo"].unique())

Sexo
Femenino     2409
Masculino    2353
F              89
M              81
Masculin       58
Nr/Otro         8
Nsnr            1
Name: count, dtype: int64
Valores no aceptables restantes: ['M' 'F' 'Masculin' 'Nr/Otro' 'Nsnr']


### **5.2** Variable Ciudad

Se revisan los valores únicos de la variable Ciudad para detectar inconsistencias de formato, como diferencias en capitalización o espacios innecesarios.


In [10]:
en_bruto["Ciudad"].unique()

array(['Bucaramanga', 'Maicao', 'Sogamoso', 'Valledupar', 'Tauramena',
       'Riohacha', 'Guateque', 'Girón', 'Yopal', 'Barrancabermeja',
       'Aguachica', 'Tibirita', 'Rioacha', 'Bosconia', 'Bbermeja',
       'Sogampso'], dtype=object)

Ambas variables, Sexo y Ciudad, presentan inconsistencias.
En el caso de sexo, tales inconsistencias fueron:
- El uso de *M* y *F* para representar a Masculino y Femenino respectivamente
- El error de tipografía *Masculin* para referirse a Masculino
- El uso de *Nsnr* (No sabe, no responde), si bien en solo un caso

En el caso de Ciudad:
- El uso de *Bbermeja* para referirse a la ciudad de Barrancabermeja
- El error ortográfico *Rioacha* para referirse a la ciudad de Riohacha
- El error de tipografía *Sogampso* para referirse a la ciudad de Sogamoso

### **5.3** Curso de Acción ante Inconsistencias 

Para la variable Ciudad, se reemplazan las inconsistencias por los valores correctos en cada caso.

Para la variable Sexo, se reemplazan M y F por los valores aceptables que representan, y el error de tipografía indicado se corrige. 
El caso particular del valor "Nsnr" (1 de 4,999 registros), no corresponde a una categoría válida de identidad de género sino a ausencia de respuesta. Por tanto, se reclasifica como valor faltante (NaN, not a number), figura que permite conservar los demás datos del registro, yproseguir con el análisis del dataset sin arrojar errores.

In [11]:
reemplazo_sexo = {
    "M": "Masculino",
    "Masculin": "Masculino",
    "F": "Femenino",
    "Nr/Otro": "NR/Otro"
}

en_bruto["Sexo"] = en_bruto["Sexo"].replace("Nsnr", np.nan) #Reemplaza el valor por NaN
en_bruto["Sexo"] = en_bruto["Sexo"].replace(reemplazo_sexo)
print(en_bruto["Sexo"].unique())

['Femenino' 'Masculino' 'NR/Otro' nan]


In [12]:
reemplazo_ciudad = {
    'Rioacha': 'Riohacha',
    'Bbermeja': 'Barrancabermeja',
    'Sogampso': 'Sogamoso',
}

en_bruto["Ciudad"] = en_bruto["Ciudad"].replace(reemplazo_ciudad)

print(en_bruto["Ciudad"].unique())

['Bucaramanga' 'Maicao' 'Sogamoso' 'Valledupar' 'Tauramena' 'Riohacha'
 'Guateque' 'Girón' 'Yopal' 'Barrancabermeja' 'Aguachica' 'Tibirita'
 'Bosconia']


## **6.** Validación de rango en variables cuantitativas

Las notas deben encontrarse en una escala de 1.0 a 5.0. 
Se verifica que no existan valores por fuera de dicho rango, los cuales indicarían errores de registro.


In [13]:
for columna in notas: #Recorre cada valor en las tres variables de las notas, captando valores fuera del rango
    fuera_rango = en_bruto[
        (en_bruto[columna] < 1.0) |
        (en_bruto[columna] > 5.0)
    ]
    
    print(f"{columna} - valores fuera de rango:", len(fuera_rango)) #Cantidad de valores fuera de rango
    print(fuera_rango[[columna]].sort_values(by=columna)) #Casos de valores fuera de rango por variable

Nota Razonamiento Cuantitativo - valores fuera de rango: 2
      Nota Razonamiento Cuantitativo
4640                            11.0
965                            328.0
Nota Comprension Lectora - valores fuera de rango: 4
      Nota Comprension Lectora
910                       15.0
4743                      19.0
96                        32.9
4687                      35.0
Nota Ingles - valores fuera de rango: 0
Empty DataFrame
Columns: [Nota Ingles]
Index: []


Se identificaron 6 registros con valores fuera del rango establecido (1.0–5.0).

Los valores detectados fueron:

- Razonamiento Cuantitativo: 11.0 y 328.0

- Comprensión Lectora: 15.0, 19.0, 32.9 y 35.0

Dichos valores exceden los límites del rango y no siguen un patrón que permita inferir una transformación válida.

Dado que representan una proporción inferior del total, y que no es posible corregirlos sin introducir supuestos arbitrarios, se procede a su eliminación.

In [14]:
condicion_valida = (
    en_bruto["Nota Razonamiento Cuantitativo"].between(1.0, 5.0) &
    en_bruto["Nota Comprension Lectora"].between(1.0, 5.0) &
    en_bruto["Nota Ingles"].between(1.0, 5.0)
)

en_bruto = en_bruto[condicion_valida]

print("Dimensiones del dataset ahora:", en_bruto.shape)

for columna in notas:
    print(columna, "mín:", en_bruto[columna].min(), "máx:", en_bruto[columna].max())

Dimensiones del dataset ahora: (4993, 7)
Nota Razonamiento Cuantitativo mín: 1.0 máx: 5.0
Nota Comprension Lectora mín: 1.0 máx: 4.8
Nota Ingles mín: 1.0 máx: 5.0


## **7.** Estado final del dataset tras la depuración

Tras el proceso de validación y limpieza, el dataset final quedó compuesto por 4.993 registros y 7 variables.

Se verificó que:

- No existen valores nulos, a excepción del valor NaN en la variable Sexo registrado en el inciso 5.3 de este notebook

- Todas las edades cumplen el requisito mínimo institucional.

- Las notas se encuentran dentro del rango 1.0–5.0.

### **7.1** El caso del valor máximo en la variable Nota Comprensión Lectora

Se observa que, mientras en Razonamiento Cuantitativo e Inglés se registran puntajes máximos de 5.0, en Comprensión Lectora el valor máximo observado es 4.8.

Este hallazgo podría estar asociado a la estructura de la evaluación o a la distribución natural de los resultados, y será explorado con mayor detalle en el notebook de análisis.

En cualquier caso, se procede a verificar que ningún valor iguala el límite superior del rango en dicha variable.

In [15]:
print("Notas iguales a 5.0:", (en_bruto["Nota Comprension Lectora"] == 5.0).sum())
en_bruto["Nota Comprension Lectora"].sort_values(ascending=False).head(5)

Notas iguales a 5.0: 0


2356    4.8
3370    4.8
4501    4.8
2002    4.8
4289    4.8
Name: Nota Comprension Lectora, dtype: float64

En efecto, ningún valor alcanza el límite superior del rango, 5.0.

Además, al ordenar los valores de forma descendente y observar los primeros cinco, se evidencia que el valor máximo de la variable Nota Comprensión Lectora sí es 4.8.

In [16]:
en_bruto.head(10)

,Sexo,Ciudad,Fecha de Nacimiento,Nota Razonamiento Cuantitativo,Nota Comprension Lectora,Nota Ingles,Edad
0,Femenino,Bucaramanga,1997-11-27,2.8,1.4,3.7,27
2,Masculino,Maicao,1998-03-11,2.4,2.0,2.0,26
3,Masculino,Sogamoso,1997-11-03,3.2,2.2,2.3,27
4,Femenino,Maicao,2001-09-07,2.5,1.5,3.2,23
5,Femenino,Valledupar,1996-12-22,1.2,2.0,1.2,28
6,Femenino,Valledupar,2004-09-22,3.6,3.7,1.5,20
7,Femenino,Valledupar,1994-08-17,4.6,3.4,2.5,30
8,Femenino,Bucaramanga,1994-07-13,2.9,2.3,1.8,30
9,Masculino,Sogamoso,1995-10-31,2.2,1.9,1.3,29
10,Femenino,Valledupar,2006-04-29,3.2,2.7,1.8,18


Por último, se almacena el dataset final en el archivo csv *datos_analisis*

In [17]:
en_bruto.to_csv('../datos/datos_analisis.csv', index=False)

---
Tras almacenar los datos ya depurados, finaliza el proceso de limpieza de datos.

Para el análisis de los datos antes mencionados, ver notebook *02_analisis_exploratorio*